In [ ]:
%load_ext autoreload
%autoreload 2

import matplotlib.pyplot as plt
import os
import pickle
import numpy as np
import pylupnt as pnt

In [ ]:
sma = 9200  # [km]
ecc = 0.60

n_planes = 2
n_planes_sat = 4

n_planes_c = 4
n_planes_sat_c = 4


wsign = 1  # w = 90 deg
f = 1  # f = 1
Omega0 = 0.0

In [ ]:
from src.constellation_design import setup_walker

et0 = pnt.convert_time(pnt.gregorian_to_time(2030, 1, 1, 12, 0, 0), pnt.UTC, pnt.TAI)

# walker 1
wsign = 1
x_walker_s = np.array([sma, ecc, wsign, n_planes, n_planes_sat, f, Omega0], dtype=float)
coes_s = setup_walker(x_walker_s, float_sma=True)

# walker 2
wsign = 0
x_walker_n = np.array([sma, ecc, wsign, n_planes, n_planes_sat, f, Omega0], dtype=float)
coes_n = setup_walker(x_walker_n, float_sma=True)

# walker 3
wsign = 0
x_walker_c = np.array(
    [sma, 0.01, wsign, n_planes_c, n_planes_sat_c, f, Omega0], dtype=float
)
coes_c = setup_walker(x_walker_c, float_sma=True)

coes = np.vstack((coes_s, coes_n, coes_c))

n_sat = coes.shape[0]

# dynamics
dynamics = pnt.NBodyDynamics()
dynamics.set_integrator(pnt.IntegratorType.RKF45)
dynamics.set_integrator_params(
    pnt.IntegratorParams(max_iter=20, abstol=1e-10, reltol=1e-10)
)
dynamics.add_body(pnt.Body.Moon(20, 20))
dynamics.add_body(pnt.Body.Earth())
dynamics.add_body(pnt.Body.Sun())
dynamics.set_time_step(60.0)  # [s]
dynamics.set_frame(pnt.MOON_CI)

# one orbit period
T_orbit = 2 * np.pi * np.sqrt(sma**3 / pnt.GM_MOON)
n_sim = int(T_orbit / 60.0)
tspan = np.linspace(0.0, T_orbit, n_sim + 1)
et = et0 + tspan
lent = len(tspan)

# propagate orbits (Moon-CI → Moon-PA)
x_orb_pa = np.zeros((n_sat, lent, 6), dtype=float)
x_orb_mci = np.zeros((n_sat, lent, 6), dtype=float)
x_orb_pa0 = np.zeros((n_sat, lent, 6), dtype=float)
for si in range(n_sat):
    rv0_op = pnt.classical_to_cart(coes[si, :], pnt.GM_MOON)
    rv0_mci = pnt.convert_frame(et0, rv0_op, pnt.MOON_OP, pnt.MOON_CI)
    x_orb_mci[si] = dynamics.propagate(rv0_mci, et0, et)
    x_orb_pa[si] = pnt.convert_frame(et, x_orb_mci[si], pnt.MOON_CI, pnt.MOON_PA)
    x_orb_pa0[si] = pnt.convert_frame(
        et0 * np.ones_like(et), x_orb_mci[si], pnt.MOON_CI, pnt.MOON_PA
    )

In [ ]:
from plotly import graph_objects as go

# Figure 2: Satellite Orbits in MCI frame ----------------------------------
fig2 = go.Figure()

w1_idx = n_planes * n_planes_sat
w2_idx = 2 * n_planes * n_planes_sat

pnt.plot.plot_orbits(fig2, x_orb_pa0[:w1_idx], color="blue")
pnt.plot.plot_orbits(fig2, x_orb_pa0[w1_idx:w2_idx], color="orange")
pnt.plot.plot_orbits(fig2, x_orb_pa0[w2_idx:], color="green")

pnt.plot.scatter(fig2, x_orb_pa0[:, 0, :3], color="red")

pnt.plot.plot_body(
    fig2,
    pnt.MOON,
    size_factor=2,
    alpha=0.5,
)
pnt.plot.set_view(fig2, -45, 20, 2.5)

fig2.show()